# 01 — Data Exploration

This notebook loads the Adamson 2016 CRISPRi Perturb-seq dataset via GEARS and explores its structure.
GEARS will auto-download the data (~200MB) on first run.

**What you'll learn:**
- What a Perturb-seq AnnData object looks like
- How perturbation conditions are structured
- How the train/val/test split works (whole perturbations held out)
- What a batch of data looks like before it enters the model

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gears import PertData
from scgpt.tokenizer.gene_tokenizer import GeneVocab

## 1. Load the Adamson dataset

GEARS manages downloading and caching. First run will take a minute.

In [ ]:
pert_data = PertData('../data')
pert_data.load(data_name='adamson')  # ~200MB download, cached after first run

adata = pert_data.adata
print(adata)

## 2. Explore the AnnData structure

`AnnData` is the standard single-cell data container:
- `adata.X` — expression matrix (cells × genes), log1p-normalized
- `adata.obs` — cell metadata (one row per cell)
- `adata.var` — gene metadata (one row per gene)

In [ ]:
print(f"Cells: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")
print(f"\nobs (cell metadata) columns: {adata.obs.columns.tolist()}")
print(f"var (gene metadata) columns: {adata.var.columns.tolist()}")

In [ ]:
# Count cells per perturbation condition
condition_counts = adata.obs['condition'].value_counts()
print(f"Unique conditions (perturbations + ctrl): {len(condition_counts)}")
print("\nTop 10 by cell count:")
print(condition_counts.head(10))
print(f"\nControl cells: {condition_counts.get('ctrl', 0)}")

In [ ]:
# Visualize cell counts per perturbation
fig, ax = plt.subplots(figsize=(12, 4))
condition_counts.iloc[1:].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Cells per perturbation (Adamson dataset)')
ax.set_xlabel('Perturbation')
ax.set_ylabel('Cell count')
ax.tick_params(axis='x', rotation=90, labelsize=6)
plt.tight_layout()
plt.show()

## 3. Check gene vocabulary overlap

scGPT was pre-trained on a 60,697-token vocabulary. Not all Adamson genes will be in the vocab — those get mapped to `<pad>` tokens and contribute no information.

In [ ]:
vocab = GeneVocab.from_file('../pretrained/scGPT_human/vocab.json')
print(f"scGPT vocab size: {len(vocab)} tokens")

genes = adata.var['gene_name'].tolist()
in_vocab = [g for g in genes if g in vocab]
not_in_vocab = [g for g in genes if g not in vocab]

print(f"\nAdamson genes: {len(genes)}")
print(f"In scGPT vocab: {len(in_vocab)} ({len(in_vocab)/len(genes)*100:.1f}%)")
print(f"Not in vocab (→ <pad>): {len(not_in_vocab)} ({len(not_in_vocab)/len(genes)*100:.1f}%)")
print(f"\nExample genes NOT in vocab: {not_in_vocab[:10]}")

## 4. Prepare the train/val/test split

**Key concept:** `split='simulation'` holds out **whole perturbations** (not random cells). This means the model must generalize to genetic perturbations it has never seen — the actual challenge.

If you held out random cells instead, a model could cheat by memorizing average expression per perturbation.

In [ ]:
pert_data.prepare_split(split='simulation', seed=1)
pert_data.get_dataloader(batch_size=64, test_batch_size=64)

# Note: pert_data.split is just the string 'sim'; use set2conditions for train/val/test lists
split = pert_data.set2conditions
print(f"Train perturbations: {len(split['train'])}")
print(f"Val perturbations:   {len(split['val'])}")
print(f"Test perturbations:  {len(split['test'])}")

# Verify no data leakage
assert len(set(split['train']) & set(split['test'])) == 0, 'DATA LEAKAGE!'
print('\nNo train/test overlap — split is clean.')

print(f"\nTest perturbations (model must predict these from scratch):\n{split['test']}")

## 5. Inspect a batch

Understanding the batch format is essential for understanding the model. Each batch is a PyTorch Geometric `Data` object.

In [ ]:
batch = next(iter(pert_data.dataloader['train_loader']))

print('Batch attributes:', [k for k in batch.keys()])
print(f"\nbatch.x shape: {batch.x.shape}")
print("  → (n_cells * n_genes, 2): flattened expression matrix")
print("  → column 0 = expression value")
print("  → column 1 = perturbation flag (1 = this gene was knocked out, 0 = control)")

print(f"\nbatch.y shape: {batch.y.shape}")
print("  → (n_cells, n_genes): target perturbed expression (what we want to predict)")

n_cells_in_batch = len(batch.y)
n_genes = adata.n_vars
assert batch.x.shape == (n_cells_in_batch * n_genes, 2), 'Unexpected batch.x shape'

print(f"\nPerturbation labels in this batch: {list(set(batch.pert))}")
print(f"Cells in this batch: {n_cells_in_batch}")

# Show the perturbation flag: 1 for the knocked-out gene, 0 for all others
pert_flags = batch.x[:, 1].view(n_cells_in_batch, n_genes)
genes_knocked_out = pert_flags[0].nonzero().flatten()
print(f"\nGenes flagged as knocked-out in cell 0: {genes_knocked_out.tolist()}")
print(f"  → gene names: {[adata.var['gene_name'].iloc[i] for i in genes_knocked_out.tolist()]}")

## 6. Look at expression distributions

Understanding the expression values helps interpret model outputs.

In [ ]:
# Expression in control cells
ctrl_adata = adata[adata.obs['condition'] == 'ctrl']
ctrl_expr = np.asarray(ctrl_adata.X.mean(axis=0)).flatten()

# Pick one test perturbation and compare to ctrl
test_pert = pert_data.set2conditions['test'][0]
pert_adata = adata[adata.obs['condition'] == test_pert]
pert_expr = np.asarray(pert_adata.X.mean(axis=0)).flatten()

delta = pert_expr - ctrl_expr

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(ctrl_expr, bins=50, color='steelblue', alpha=0.7)
axes[0].set_title('Control mean expression (all genes)')
axes[0].set_xlabel('log1p expression')

axes[1].hist(pert_expr, bins=50, color='orange', alpha=0.7)
axes[1].set_title(f'Perturbed mean expression\n({test_pert})')
axes[1].set_xlabel('log1p expression')

axes[2].hist(delta, bins=50, color='green', alpha=0.7)
axes[2].axvline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title("Delta (perturbed - ctrl)\nMost genes don't change!")
axes[2].set_xlabel('Delta expression')

print(f"Genes with |delta| > 0.5: {(np.abs(delta) > 0.5).sum()} / {len(delta)}")
print(f"This is why pearson_de (DE genes only) is the informative metric.")

plt.tight_layout()
plt.show()

## Phase 2 Validation Checks

In [ ]:
assert adata.shape[0] > 4000, 'Expected ~5K cells'
assert len(adata.obs['condition'].unique()) > 80, 'Expected ~88 conditions'
assert 'ctrl' in adata.obs['condition'].values
assert len(split['test']) > 0, 'Test split is empty'
assert len(set(split['train']) & set(split['test'])) == 0, 'Data leakage!'
assert batch.x.shape[1] == 2, 'batch.x should have 2 cols'

print('All Phase 2 validation checks passed!')
print(f'  Cells: {adata.shape[0]}, Genes: {adata.shape[1]}')
print(f'  Conditions: {len(adata.obs["condition"].unique())}')
print(f'  Split: {len(split["train"])} train / {len(split["val"])} val / {len(split["test"])} test')